In [1]:
!pip install -q timm torch torchvision scikit-learn pandas pillow tqdm kaggle

In [2]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from torch.utils.data import DataLoader, Subset
import numpy as np
import torch
import torch.nn as nn
import timm
import torch.optim as optim
from tqdm import tqdm
from torch.amp import autocast, GradScaler
import os

# Dataset

In [3]:
import os
from google.colab import userdata

os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')

import kagglehub

path = kagglehub.dataset_download(
    "salviohexia/isic-2019-skin-lesion-images-for-classification"
)

print("Dataset path:", path)

Using Colab cache for faster access to the 'isic-2019-skin-lesion-images-for-classification' dataset.
Dataset path: /kaggle/input/isic-2019-skin-lesion-images-for-classification


## Custom Dataset Class

In [4]:
from torch.utils.data import Dataset
import os
from PIL import Image


class ISICDataset(Dataset):
    def __init__(self, root, transform=None):
        self.root = root
        self.transform = transform

        self.class_names = sorted([
            d for d in os.listdir(root)
            if os.path.isdir(os.path.join(root, d))
        ])

        self.class_to_idx = {cls: i for i, cls in enumerate(self.class_names)}

        self.samples = []
        for cls in self.class_names:
            cls_path = os.path.join(root, cls)
            label = self.class_to_idx[cls]

            for img_name in os.listdir(cls_path):
                if img_name.lower().endswith((".jpg", ".png", ".jpeg")):
                    self.samples.append((os.path.join(cls_path, img_name), label))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]

        img = Image.open(path).convert("RGB")

        if self.transform:
            img = self.transform(img)

        return img, label

# Data Augmentation & Preprocessing

In [5]:
import torchvision.transforms as T

IMG_SIZE = 465

train_tf = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.RandomHorizontalFlip(),
    T.RandomVerticalFlip(),  # NEW
    T.RandomRotation(15),    # slightly stronger
    T.ColorJitter(0.3, 0.3, 0.3),  # stronger
    T.RandomAffine(degrees=0, translate=(0.05, 0.05)),  # NEW
    T.ToTensor(),
    T.Normalize([0.5] * 3, [0.5] * 3)
])

val_tf = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize([0.5] * 3, [0.5] * 3)
])

In [6]:
import random

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# Dataset Splitting

In [7]:
DATA_PATH = path
base_dataset = ISICDataset(path, transform=None)
num_classes = len(base_dataset.class_names)

labels = np.array([label for _, label in base_dataset.samples])
indices = np.arange(len(labels))

train_idx, val_idx = train_test_split(
    indices, test_size=0.2, random_state=42, stratify=labels
)

train_dataset = ISICDataset(root=path, transform=train_tf)
val_dataset = ISICDataset(root=path, transform=val_tf)

train_ds = Subset(train_dataset, train_idx)
val_ds = Subset(val_dataset, val_idx)

print(len(train_ds), len(val_ds))

20264 5067


# Class Imbalance Analysis

In [8]:

train_labels = [base_dataset.samples[i][1] for i in train_idx]
class_counts = np.bincount(train_labels, minlength=num_classes)

print("\n=== Training class distribution ===")
for i, cls in enumerate(base_dataset.class_names):
    print(f"{cls:20s}: {class_counts[i]:6d}  ({class_counts[i] / class_counts.sum() * 100:.2f}%)")
print(f"Imbalance ratio (max/min): {class_counts.max() / class_counts.min():.1f}x")



=== Training class distribution ===
AK                  :    694  (3.42%)
BCC                 :   2658  (13.12%)
BKL                 :   2099  (10.36%)
DF                  :    191  (0.94%)
MEL                 :   3618  (17.85%)
NV                  :  10300  (50.83%)
SCC                 :    502  (2.48%)
VASC                :    202  (1.00%)
Imbalance ratio (max/min): 53.9x


## Handling Class Imbalance

In [9]:
from torch.utils.data import WeightedRandomSampler


USE_SAMPLER = True
USE_FOCAL_LOSS = True

class_weights_for_sampler = 1.0 / np.maximum(class_counts, 1)
sample_weights = [class_weights_for_sampler[label] for label in train_labels]

sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True,
)



# DataLoaders

In [10]:
train_loader = DataLoader(
    train_ds,
    batch_size=32,
    sampler=sampler if USE_SAMPLER else None,
    shuffle=False if USE_SAMPLER else True,  # sampler and shuffle are mutually exclusive
    num_workers=2,
    pin_memory=True,
)

val_loader = DataLoader(
    val_ds,
    batch_size=32,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
)


# Loss Functions

In [11]:
import torch.nn.functional as F

class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, weight=None, label_smoothing=0.0):
        super().__init__()
        self.gamma = gamma
        self.weight = weight
        self.label_smoothing = label_smoothing

    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(
            inputs, targets, weight=self.weight,
            label_smoothing=self.label_smoothing, reduction="none"
        )
        pt = torch.exp(-ce_loss)
        focal_loss = ((1 - pt) ** self.gamma) * ce_loss
        return focal_loss.mean()


In [12]:

raw_weights = class_counts.sum() / (class_counts + 1e-6)
class_weights = torch.tensor(
    raw_weights / raw_weights.mean(), dtype=torch.float
).to(device)

print("\n--- class_weights (mean-normalized) ---")
for i, cls in enumerate(base_dataset.class_names):
    print(f"{cls:20s}: {class_weights[i].item():.3f}")

if USE_FOCAL_LOSS:
    criterion = FocalLoss(gamma=1.5, weight=class_weights, label_smoothing=0.05)
else:
    criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.1)


--- class_weights (mean-normalized) ---
AK                  : 0.777
BCC                 : 0.203
BKL                 : 0.257
DF                  : 2.821
MEL                 : 0.149
NV                  : 0.052
SCC                 : 1.073
VASC                : 2.668


# Model Definition

In [13]:

def build_model(model_name="efficientnet_b5"):
    model = timm.create_model(model_name, pretrained=True)
    model.classifier = nn.Linear(model.classifier.in_features, num_classes)

    # Freeze everything
    for p in model.parameters():
        p.requires_grad = False

    # Unfreeze last blocks + classifier (fine-tuning)
    for name, p in model.named_parameters():
        if any(b in name for b in ["blocks.4", "blocks.5", "blocks.6", "blocks.7", "classifier"]):
            p.requires_grad = True

    return model.to(device)



# Optimizer & Scheduler

In [14]:
import torch.optim as optim

model = build_model("efficientnet_b5")

optimizer = optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=5e-5
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=15
)

model.safetensors:   0%|          | 0.00/122M [00:00<?, ?B/s]

# Training Pipeline

In [15]:
import os
import time
import torch
from tqdm import tqdm
from sklearn.metrics import f1_score, classification_report, confusion_matrix, recall_score
from torch.amp import autocast, GradScaler


def train_model(
    model, optimizer, criterion, train_loader, val_loader, device,
    epochs, name, base_dataset, img_size, scheduler=None, patience=4
):
    scaler = GradScaler(device="cuda")

    best_f1 = 0.0
    best_state = None
    counter = 0

    os.makedirs("checkpoints", exist_ok=True)

    # ----------- MODEL SIZE (once) -----------
    num_params = sum(p.numel() for p in model.parameters())
    print(f"\nModel params: {num_params/1e6:.2f}M")

    for epoch in range(epochs):
        start_time = time.time()

        model.train()
        total_loss = 0.0

        for imgs, labels in tqdm(train_loader, desc=f"{name} Epoch {epoch+1} [Train]"):
            imgs, labels = imgs.to(device), labels.to(device)

            optimizer.zero_grad(set_to_none=True)

            with autocast(device_type="cuda"):
                outputs = model(imgs)
                loss = criterion(outputs, labels)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            total_loss += loss.item()

        avg_loss = total_loss / len(train_loader)

        if scheduler is not None:
            scheduler.step()

        # ----------- VALIDATION -----------
        model.eval()
        preds_all, labels_all = [], []

        with torch.no_grad():
            for imgs, labels in tqdm(val_loader, desc=f"{name} Epoch {epoch+1} [Val]"):
                imgs, labels = imgs.to(device), labels.to(device)

                with autocast(device_type="cuda"):
                    outputs = model(imgs)

                preds = torch.argmax(outputs, dim=1)

                preds_all.extend(preds.cpu().tolist())
                labels_all.extend(labels.cpu().tolist())

        # ----------- METRICS -----------
        f1 = f1_score(labels_all, preds_all, average="macro")
        class_f1 = f1_score(labels_all, preds_all, average=None, zero_division=0)

        recall_macro = recall_score(labels_all, preds_all, average="macro")
        recall_per_class = recall_score(labels_all, preds_all, average=None, zero_division=0)

        epoch_time = time.time() - start_time

        print(f"\n{name} Epoch {epoch+1}")
        print("Loss:", avg_loss)
        print("Macro F1:", f1)
        print("Macro Recall:", recall_macro)
        print("Epoch time:", f"{epoch_time:.2f} sec")
        print("LR:", scheduler.get_last_lr()[0] if scheduler else "n/a")

        print("\nPer-class F1:")
        for i, f1c in enumerate(class_f1):
            print(f"{base_dataset.class_names[i]:20s}: {f1c:.4f}")

        print("\nPer-class Recall:")
        for i, r in enumerate(recall_per_class):
            print(f"{base_dataset.class_names[i]:20s}: {r:.4f}")

        # ----------- INFERENCE SPEED -----------
        start_inf = time.time()
        with torch.no_grad():
            for imgs, _ in val_loader:
                imgs = imgs.to(device)
                _ = model(imgs)
        inference_time = time.time() - start_inf
        print(f"Inference time (val set): {inference_time:.2f} sec")

        # ----------- FINAL METRICS -----------
        if epoch == epochs - 1:
            print("\nClassification Report:\n")
            print(classification_report(
                labels_all, preds_all,
                target_names=base_dataset.class_names, digits=4
            ))

            print("\nConfusion Matrix:\n")
            print(confusion_matrix(labels_all, preds_all))

        # ----------- SAVE BEST -----------
        if f1 > best_f1:
            best_f1 = f1
            best_state = model.state_dict()
            counter = 0

            torch.save({
                "model_name": name,
                "epoch": epoch + 1,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "scheduler_state_dict": scheduler.state_dict() if scheduler else None,
                "loss": avg_loss,
                "val_f1": f1,
                "class_names": base_dataset.class_names,
                "class_to_idx": base_dataset.class_to_idx,
                "input_size": img_size
            }, f"checkpoints/{name}_best.pth")

            print("Best model updated")

        else:
            counter += 1
            print(f"No improvement ({counter}/{patience})")

        if counter >= patience:
            print("Early stopping triggered")
            break

    # ----------- LOAD BEST -----------
    if best_state is not None:
        model.load_state_dict(best_state)

    # ----------- FINAL SAVE -----------
    import json

    save_path = f"checkpoints/{name}_final.pth"

    torch.save({
        "model_name": name,
        "model_state_dict": model.state_dict(),
        "class_names": base_dataset.class_names,
        "class_to_idx": base_dataset.class_to_idx,
        "input_size": img_size,
        "best_f1": best_f1
    }, save_path)

    # -------- SAVE CONFIG (VERY IMPORTANT) --------
    config = {
        "model_name": name,
        "architecture": "efficientnet_b5",
        "num_classes": len(base_dataset.class_names),
        "class_names": base_dataset.class_names,
        "image_size": img_size,
        "best_f1": best_f1
    }

    with open("checkpoints/config.json", "w") as f:
        json.dump(config, f, indent=4)

    # -------- SAVE METRICS --------
    metrics = {
        "best_f1": best_f1,
        "macro_f1": float(f1),
        "macro_recall": float(recall_macro)
    }

    with open("checkpoints/metrics.json", "w") as f:
        json.dump(metrics, f, indent=4)

    return best_f1

# Training

In [16]:
criterion = nn.CrossEntropyLoss()
img_size=456
best_f1 = train_model(
    model=model,
    optimizer=optimizer,
    criterion=criterion,
    train_loader=train_loader,
    val_loader=val_loader,
    device=device,
    epochs=10,
    name="efficientnet_b5_ft",
    base_dataset=base_dataset,
    img_size=img_size,
    scheduler=scheduler,
    patience=4
)


Model params: 28.36M


efficientnet_b5_ft Epoch 1 [Val]: 100%|██████████| 159/159 [01:58<00:00,  1.34it/s]


efficientnet_b5_ft Epoch 1
Loss: 0.7559775753117134
Macro F1: 0.6734042155792532
Macro Recall: 0.7701853452286762
Epoch time: 800.51 sec
LR: 4.9453690018345144e-05

Per-class F1:
AK                  : 0.5572
BCC                 : 0.7924
BKL                 : 0.6240
DF                  : 0.7647
MEL                 : 0.6339
NV                  : 0.8345
SCC                 : 0.5852
VASC                : 0.5952

Per-class Recall:
AK                  : 0.7457
BCC                 : 0.8436
BKL                 : 0.6229
DF                  : 0.8125
MEL                 : 0.6704
NV                  : 0.7639
SCC                 : 0.7222
VASC                : 0.9804


Inference time (val set): 111.18 sec
Best model updated


efficientnet_b5_ft Epoch 2 [Val]: 100%|██████████| 159/159 [01:47<00:00,  1.48it/s]

...


efficientnet_b5_ft Epoch 9
Loss: 0.062019453140134316
Macro F1: 0.8465008562776668
Macro Recall: 0.8330593001115498
Epoch time: 746.57 sec
LR: 1.727457514062632e-05

Per-class F1:
AK                  : 0.7449
BCC                 : 0.9091
BKL                 : 0.8155
DF                  : 0.8636
MEL                 : 0.7922
NV                  : 0.9235
SCC                 : 0.8033
VASC                : 0.9200

Per-class Recall:
AK                  : 0.7341
BCC                 : 0.9474
BKL                 : 0.8038
DF                  : 0.7917
MEL                 : 0.7843
NV                  : 0.9235
SCC                 : 0.7778
VASC                : 0.9020


Inference time (val set): 111.85 sec
Best model updated


efficientnet_b5_ft Epoch 10 [Val]: 100%|██████████| 159/159 [01:46<00:00,  1.49it/s]


efficientnet_b5_ft Epoch 10
Loss: 0.05095719720004861
Macro F1: 0.8389944381252128
Macro Recall: 0.8183662393974961
Epoch time: 747.40 sec
LR: 1.2500000000000009e-05

Per-class F1:
AK                  : 0.7409
BCC                 : 0.9176
BKL                 : 0.8141
DF                  : 0.8636
MEL                 : 0.7867
NV                  : 0.9269
SCC                 : 0.7778
VASC                : 0.8842

Per-class Recall:
AK                  : 0.7688
BCC                 : 0.9383
BKL                 : 0.8343
DF                  : 0.7917
MEL                 : 0.7201
NV                  : 0.9480
SCC                 : 0.7222
VASC                : 0.8235


Inference time (val set): 110.73 sec

Classification Report:

              precision    recall  f1-score   support

          AK     0.7151    0.7688    0.7409       173
         BCC     0.8978    0.9383    0.9176       665
         BKL     0.7949    0.8343    0.8141       525
          DF     0.9500    0.7917    0.8636        48
         MEL     0.8668    0.7201    0.7867       904
          NV     0.9068    0.9480    0.9269      2575
         SCC     0.8426    0.7222    0.7778       126
        VASC     0.9545    0.8235    0.8842        51

    accuracy                         0.8798      5067
   macro avg     0.8661    0.8184    0.8390      5067
weighted avg     0.8796    0.8798    0.8779      5067


Confusion Matrix:

[[ 133   15   17    1    2    0    5    0]
 [  15  624    8    0    4   11    2    1]
 [  10    9  438    0   17   44    7    0]
 [   1    3    1   38    0    4    0    1]
 [  11   19   35    0  651  187    1    0]
 [   3   17   38    1   73 2441    2    0]
 [  13   

# Hugging Face Upload

In [17]:
from google.colab import userdata
from huggingface_hub import login

HF_TOKEN = userdata.get("HF_TOKEN")

login(HF_TOKEN)

from huggingface_hub import create_repo, upload_folder

repo_id = "menna143/skin-classifier-EfficientNet-B5"

create_repo(repo_id, exist_ok=True)

upload_folder(
    repo_id=repo_id,
    folder_path="checkpoints",
    path_in_repo="",
    commit_message="Clinicore EfficientNet-B5 model upload"
)

print("Model pushed to Hugging Face")


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...ficientnet_b5_ft_best.pth:   2%|1         | 5.71MB /  316MB            

  ...icientnet_b5_ft_final.pth:   5%|4         | 5.70MB /  114MB            

Model pushed to Hugging Face
